# Residential Property Rental Demand Analysis
**EDAV Project Notebook**

**Student:** J.Aravind  
**Roll No.:** 160125737088  
**Department:** Information Technology  
**Faculty:** Mr. G. Srikanth  

## Project overview
This notebook explores rental demand for residential properties using property features, monthly rent, inquiry counts, days listed, and occupancy status. It demonstrates data loading, cleaning, descriptive statistics, grouping, visualization, and simple interpretation.

> **Dataset note:** The attached CSV is a synthetic demonstration dataset created for learning and notebook execution. It is not real estate market data and must not be presented as an actual survey or current market estimate.


## 1. Import libraries
We use pandas to manage tabular data, NumPy for numerical operations, Matplotlib and Seaborn to create charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the dataset
Keep the CSV file in the same folder as this notebook. The preview helps confirm that the file loaded correctly.

In [ ]:
df = pd.read_csv("residential_property_rental_demand_demo_dataset.csv")
print("Dataset shape:", df.shape)
df.head()


## 3. Understand the columns
Each row represents one demonstration property listing. The columns describe its location, size, rent, inquiries, listing duration, and occupancy.

In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


## 4. Check data quality
We check missing values, duplicate records, and data types before analysis. This sample dataset is generated with complete rows, but the checks are useful for real datasets too.

In [ ]:
print("Missing values by column:")
display(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())


## 5. Prepare the data
Remove duplicate rows if any are found and convert category-like columns to consistent text values. This makes later grouping more reliable.

In [ ]:
df = df.drop_duplicates().copy()
for col in ["City", "Locality", "Property_Type", "Furnishing", "Occupied"]:
    df[col] = df[col].astype("string").str.strip()

df["Monthly_Rent_INR"] = pd.to_numeric(df["Monthly_Rent_INR"], errors="coerce")
df["Monthly_Inquiries"] = pd.to_numeric(df["Monthly_Inquiries"], errors="coerce")
df["Days_Listed"] = pd.to_numeric(df["Days_Listed"], errors="coerce")
df = df.dropna(subset=["Monthly_Rent_INR", "Monthly_Inquiries", "Days_Listed"])
df.head()


## 6. Summary of rental demand
We use monthly inquiries as a simple indicator of interest in a listing. It is only a proxy for demand; inquiries do not guarantee that a tenant will rent the property.

In [ ]:
print("Average monthly rent (INR):", round(df["Monthly_Rent_INR"].mean(), 2))
print("Median monthly rent (INR):", round(df["Monthly_Rent_INR"].median(), 2))
print("Average monthly inquiries:", round(df["Monthly_Inquiries"].mean(), 2))
print("Average days listed:", round(df["Days_Listed"].mean(), 2))
print("Occupied share (%):", round((df["Occupied"].eq("Yes").mean()) * 100, 2))


## 7. Compare average rent by city
The bar chart compares average rent in the cities represented in the sample. Differences in this synthetic dataset are illustrative only.

In [ ]:
city_rent = df.groupby("City", as_index=False)["Monthly_Rent_INR"].mean().sort_values("Monthly_Rent_INR", ascending=False)
display(city_rent)

plt.figure(figsize=(9, 5))
sns.barplot(data=city_rent, x="City", y="Monthly_Rent_INR")
plt.title("Average Monthly Rent by City")
plt.xlabel("City")
plt.ylabel("Average monthly rent (INR)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 8. Rental demand by property size
We compare the average number of monthly inquiries for different bedroom counts. A larger property may appeal to a different group of tenants, so the pattern should be interpreted alongside rent and locality.

In [ ]:
bedroom_demand = df.groupby("Bedrooms", as_index=False)["Monthly_Inquiries"].mean()
display(bedroom_demand)

plt.figure(figsize=(8, 5))
sns.barplot(data=bedroom_demand, x="Bedrooms", y="Monthly_Inquiries")
plt.title("Average Monthly Inquiries by Number of Bedrooms")
plt.xlabel("Bedrooms")
plt.ylabel("Average monthly inquiries")
plt.tight_layout()
plt.show()


## 9. Rent and inquiry relationship
The scatter plot helps inspect whether listings with different rents receive different levels of interest. The relationship may be influenced by city, property size, furnishing, and other factors.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Monthly_Rent_INR", y="Monthly_Inquiries", hue="City", alpha=0.75)
plt.title("Monthly Rent vs. Monthly Inquiries")
plt.xlabel("Monthly rent (INR)")
plt.ylabel("Monthly inquiries")
plt.tight_layout()
plt.show()


## 10. Days listed and inquiry levels
This chart explores whether listings receiving more inquiries tend to remain listed for fewer days. This is descriptive analysis, not proof of cause and effect.

In [ ]:
plt.figure(figsize=(9, 5))
sns.regplot(data=df, x="Monthly_Inquiries", y="Days_Listed", scatter_kws={"alpha": 0.45}, line_kws={"color": "red"})
plt.title("Monthly Inquiries vs. Days Listed")
plt.xlabel("Monthly inquiries")
plt.ylabel("Days listed")
plt.tight_layout()
plt.show()


## 11. Furnishing and rental prices
Furnishing may be associated with rent, but property size, age, and locality also matter. The box plot shows the spread of rents in each furnishing category.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Furnishing", y="Monthly_Rent_INR")
plt.title("Monthly Rent by Furnishing Status")
plt.xlabel("Furnishing")
plt.ylabel("Monthly rent (INR)")
plt.tight_layout()
plt.show()


## 12. Occupancy status
We count properties marked as occupied and not occupied in the demonstration data. Occupancy is a simple outcome field and may depend on factors not included in this dataset.

In [ ]:
occupancy_counts = df["Occupied"].value_counts().rename_axis("Occupied").reset_index(name="Properties")
display(occupancy_counts)

plt.figure(figsize=(6, 5))
sns.barplot(data=occupancy_counts, x="Occupied", y="Properties")
plt.title("Properties by Occupancy Status")
plt.xlabel("Occupied")
plt.ylabel("Number of properties")
plt.tight_layout()
plt.show()


## 13. Group by city and property type
This table compares average rent and average inquiry counts for each city and property type combination. Grouping helps summarize a dataset with many individual listings.

In [ ]:
city_type_summary = (
    df.groupby(["City", "Property_Type"], as_index=False)
      .agg(
          Average_Rent_INR=("Monthly_Rent_INR", "mean"),
          Average_Inquiries=("Monthly_Inquiries", "mean"),
          Average_Days_Listed=("Days_Listed", "mean"),
          Listing_Count=("Property_ID", "count")
      )
)
display(city_type_summary.round(2))


## 14. Monthly listing pattern
The listing month column is a demonstration field. The chart can be used to practise monthly grouping, but it does not represent verified seasonal rental trends.

In [ ]:
monthly_listings = df.groupby("Listing_Month").size().reindex(range(1, 13), fill_value=0)
display(monthly_listings.rename("Listing_Count").to_frame())

plt.figure(figsize=(9, 5))
monthly_listings.plot(kind="line", marker="o")
plt.title("Number of Listings by Month")
plt.xlabel("Listing month")
plt.ylabel("Number of listings")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()


## 15. Correlation among numeric variables
Correlation describes how numeric variables move together in this sample. Correlation does not establish causation, and it may be affected by the way demonstration data was generated.

In [ ]:
numeric_cols = [
    "Bedrooms", "Area_sqft", "Building_Age_years",
    "Distance_to_City_Centre_km", "Monthly_Rent_INR",
    "Monthly_Inquiries", "Days_Listed", "Listing_Month"
]
corr = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numeric Variables")
plt.tight_layout()
plt.show()


## 16. Identify listings with high inquiry counts
This step selects the top listings by monthly inquiries. These are examples for further investigation, not a claim that the properties are real or available for rent.

In [ ]:
top_demand = df.sort_values("Monthly_Inquiries", ascending=False).head(10)
top_demand[[
    "Property_ID", "City", "Locality", "Property_Type", "Bedrooms",
    "Monthly_Rent_INR", "Monthly_Inquiries", "Days_Listed", "Occupied"
]]


## 17. Simple findings
Use the calculated values below to describe the sample in your own words. Since the data is synthetic, avoid presenting these numbers as actual market findings.

In [ ]:
highest_rent_city = city_rent.iloc[0]
lowest_rent_city = city_rent.iloc[-1]
highest_demand_bedrooms = bedroom_demand.sort_values("Monthly_Inquiries", ascending=False).iloc[0]

print(f"Highest average rent in this sample: {highest_rent_city['City']} "
      f"(INR {highest_rent_city['Monthly_Rent_INR']:.0f}).")
print(f"Lowest average rent in this sample: {lowest_rent_city['City']} "
      f"(INR {lowest_rent_city['Monthly_Rent_INR']:.0f}).")
print(f"Bedroom category with highest average inquiries: "
      f"{int(highest_demand_bedrooms['Bedrooms'])} bedroom(s).")


## 18. Conclusion
This notebook demonstrates how exploratory data analysis can be used to study residential rental listings. Grouping and charts help compare rents across cities, inspect inquiry levels, review listing duration, and explore occupancy.

The results are limited by the demonstration dataset. A real study should use verified listing records collected over a defined period, document the source and collection method, remove duplicate or misleading listings, and consider factors such as locality, property condition, transport access, lease terms, and changes over time.

### Possible improvements
- Use real, permissioned rental listing data.
- Include the date of collection and the source of each listing.
- Compare demand across neighbourhoods and months.
- Investigate outliers and validate unusual rent values.
- Avoid causal claims unless supported by an appropriate study design.


## Appendix: Data dictionary

| Column | Meaning |
|---|---|
| Property_ID | Unique identifier for a sample listing |
| City / Locality | Location fields |
| Property_Type | Apartment, independent house, or studio |
| Bedrooms | Number of bedrooms |
| Area_sqft | Approximate property area in square feet |
| Furnishing | Furnishing category |
| Building_Age_years | Approximate building age |
| Distance_to_City_Centre_km | Distance in kilometres |
| Monthly_Rent_INR | Monthly rent in Indian rupees |
| Monthly_Inquiries | Demonstration count of monthly inquiries |
| Days_Listed | Number of days listed |
| Occupied | Occupancy status |
| Listing_Month | Month number from 1 to 12 |

**Submission note:** Run the notebook from top to bottom and keep the CSV in the same folder. All generated values in the included dataset are synthetic.
